# California Birds — Data Check Notebook

This notebook checks whether the dataset can be read correctly, whether class names are normal, and whether image transforms work as expected.


In [ ]:
import os
import random
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets

DATA_ROOT = '/Users/wangyiding/ML_Model_CaliforniaBirds/data'
IMAGE_SIZE = 224
SEED = 42
random.seed(SEED)


In [ ]:
dataset = datasets.ImageFolder(root=DATA_ROOT)
class_names = dataset.classes
samples = dataset.samples

print('Total classes:', len(class_names))
print('Total images :', len(samples))
print('First 10 classes:')
for name in class_names[:10]:
    print(' -', name)


In [ ]:
label_counts = Counter([label for _, label in samples])
counts = list(label_counts.values())

print('Min images per class:', min(counts))
print('Max images per class:', max(counts))
print('Average images per class:', sum(counts) / len(counts))


In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(counts, bins=20)
plt.title('Images Per Class Distribution')
plt.xlabel('Images per class')
plt.ylabel('Number of classes')
plt.show()


In [ ]:
def show_random_images(dataset_obj, num_images=9):
    chosen = random.sample(dataset_obj.samples, num_images)
    plt.figure(figsize=(12, 12))
    for i, (img_path, label) in enumerate(chosen, start=1):
        image = Image.open(img_path).convert('RGB')
        plt.subplot(3, 3, i)
        plt.imshow(image)
        plt.title(dataset_obj.classes[label], fontsize=9)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_random_images(dataset, num_images=9)


In [ ]:
import sys
PROJECT_ROOT = '/Users/wangyiding/ML_Model_CaliforniaBirds'
if PROJECT_ROOT not in sys.path:
    sys.path.append(os.path.join(PROJECT_ROOT, 'src'))

from dataset import build_transforms

train_transform, val_transform = build_transforms(image_size=IMAGE_SIZE)
print('Transforms loaded successfully.')


In [ ]:
def show_transform_effect(dataset_obj):
    img_path, label = random.choice(dataset_obj.samples)
    original = Image.open(img_path).convert('RGB')
    transformed = train_transform(original)

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title('Original')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(transformed.permute(1, 2, 0))
    plt.title('Train Transform')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_transform_effect(dataset)


In [ ]:
from dataset import build_dataloaders

train_loader, val_loader, class_names = build_dataloaders(
    data_root=DATA_ROOT,
    batch_size=8,
    image_size=IMAGE_SIZE,
    val_split=0.1,
    num_workers=0,
    max_samples=500,
    seed=42,
)

print('Train batches:', len(train_loader))
print('Val batches  :', len(val_loader))
print('Returned classes:', len(class_names))


In [ ]:
images, labels = next(iter(train_loader))
print('Batch image shape:', images.shape)
print('Batch label shape:', labels.shape)
print('First 8 label ids:', labels.tolist())


## Recommended Use

Run this notebook before training whenever you change dataset location, transforms, or split logic.
